Еженедельный отчет о реализациии из раздела документы и Уведомление о выкупе

1. Получаю их с ВБ

- По эндпоинту https://documents-api.wildberries.ru/api/v1/documents/list
получаю информацию о документах с их реквизитами.



- По эндпоинту https://documents-api.wildberries.ru/api/v1/documents/download/all скачиваю их

In [2]:
import aiohttp
import asyncio
from datetime import datetime, timedelta
import zipfile
import io
import base64
from openpyxl import load_workbook


# Импортируем библиотеки для работы с системой и путями
import os
from  pathlib import Path
import sys
# Установка базовой директории и пути к файлу с учетными данными. Используем конструкцию try-except для обработки возможных ошибок при определении пути для notebook.
try:
    BASE_DIR = Path(__file__).resolve().parents[2]
except NameError:
    BASE_DIR = Path.cwd().resolve().parents[1] 

SRC_DIR = os.path.join(BASE_DIR, 'src')

# Добавляем src в пути поиска модулей
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# Импорт самописных утилит
from core.utils_general import load_api_tokens

In [3]:
async def get_doc_list_wb(
    account: str,
    token: str,
    session: aiohttp.ClientSession,
    date_from=datetime.now() - timedelta(days=28),
    date_to=datetime.now() - timedelta(days=1),
    limit=50
):
    """Возвращает список всех документов за период для двух категорий."""
    doc_names = ('weekly-implementation-report', 'redeem-notification')
    all_documents = []
    url = "https://documents-api.wildberries.ru/api/v1/documents/list"
    headers = {"Authorization": token}
    delay = 10
    attempts = 10

    for category in doc_names:
        offset = 0
        while True:
            params = {
                "beginTime": date_from.strftime("%Y-%m-%d"),
                "endTime": date_to.strftime("%Y-%m-%d"),
                "category": category,
                "limit": limit,
                "offset": offset
            }

            for attempt in range(attempts + 1):
                try:
                    async with session.get(url, headers=headers, params=params, timeout=60) as res:
                        if res.status == 200:
                            data = await res.json()
                            documents = data.get('data', {}).get('documents', [])
                            if not documents:
                                break  # Выход из while, документов больше нет
                            for doc in documents:
                                doc['account'] = account
                                doc['doc_type'] = category  # добавляем метку категории
                            all_documents.extend(documents)
                            offset += limit  # переходим к следующей странице
                            break  # успех, выходим из цикла попыток
                        else:
                            error_detail = await res.text() if res.content_type != 'application/json' else (await res.json()).get('detail', '')
                            if res.status in (429, 503):
                                print(f"⏳ [{account}] Ошибка {res.status}: {error_detail}. Повтор через {delay} сек.")
                                await asyncio.sleep(delay)
                                delay *= 2
                                continue  # повторяем попытку с тем же offset
                            elif res.status == 401:
                                print(f"🔑 [{account}] Ошибка 401: Неверный токен. ({error_detail})")
                                break  # нет смысла повторять
                            elif res.status == 400:
                                print(f"❓ [{account}] Ошибка 400: Плохой запрос. ({error_detail})")
                                break
                            else:
                                print(f"❌ [{account}] Ошибка {res.status}: {error_detail}")
                                break
                except (aiohttp.ClientError, asyncio.TimeoutError) as e:
                    print(f"💥 [{account}] Ошибка сети: {e}. Попытка {attempt+1}/{attempts+1}")
                    if attempt == attempts:
                        print(f"❌ [{account}] Превышено число попыток для категории {category}, offset={offset}")
                        break
                    await asyncio.sleep(delay)
                except Exception as e:
                    print(f"💥 [{account}] Непредвиденная ошибка: {e}")
                    break

            else:
                # Цикл попыток исчерпан без успеха – прерываем получение этой категории
                print(f"❌ [{account}] Не удалось получить данные для категории {category}")
                break

            # Проверяем, нужно ли продолжать while
            if not documents:  # условие выхода из while
                break

    return all_documents

In [4]:
async def fetch_doc_list_wb(tokens: dict = load_api_tokens()):
    """Функция для получения данных о документах по всем ЛК"""
    # Асинхронно обрабатываем все аккаунты и токены
    async with aiohttp.ClientSession() as session:
        # Создаем задачи для каждого аккаунта и токена
        tasks = [get_doc_list_wb(account, token, session) for account, token in tokens.items()]
        # Ожидаем завершения всех задач и собираем результаты        
        results = await asyncio.gather(*tasks)
        return results

In [5]:
doc_list = await fetch_doc_list_wb()
doc_list

[[{'serviceName': 'weekly-implementation-report-628078633',
   'name': 'Отчет №628078633 от 2026-02-09',
   'category': 'Weekly implementation report',
   'extensions': ['zip'],
   'creationTime': '2026-02-17T16:40:49Z',
   'viewed': False,
   'account': 'Даниелян',
   'doc_type': 'weekly-implementation-report'},
  {'serviceName': 'weekly-implementation-report-620986773',
   'name': 'Отчет №620986773 от 2026-02-02',
   'category': 'Weekly implementation report',
   'extensions': ['zip'],
   'creationTime': '2026-02-09T13:36:03Z',
   'viewed': False,
   'account': 'Даниелян',
   'doc_type': 'weekly-implementation-report'},
  {'serviceName': 'weekly-implementation-report-615070057',
   'name': 'Отчет №615070057 от 2026-01-26',
   'category': 'Weekly implementation report',
   'extensions': ['zip'],
   'creationTime': '2026-02-03T13:13:27Z',
   'viewed': False,
   'account': 'Даниелян',
   'doc_type': 'weekly-implementation-report'},
  {'serviceName': 'redeem-notification-628078646',
   '

In [ ]:
class ReqWB:
    def __init__(self):
        self.docs = None
    async def check(self):
        if self.docs is None:
            self.docs = await fetch_doc_list_wb(load_api_tokens())



In [ ]:
async def processing_doc_list_wb(data=None)-> dict:
    """Функция получает и обрабатывает список доступных документов, возвращая словарь со списком документов, доступных для дальнейшего скачивания. 
    
    - data = await fetch_doc_list_wb(load_api_tokens()) возвращает список всех доступных для скачивания документов со всех аккаунтов

    """

    if data is None:
        data = await fetch_doc_list_wb(load_api_tokens())

    doc_dict = {}

    for first_level in data:
        if first_level:
            for doc in first_level:
                account = doc['account']
                
                if account not in doc_dict:
                    doc_dict[account] = []
                
                processed_doc = doc.copy()
                
                if processed_doc.get('extensions'):
                    processed_doc['extension'] = processed_doc['extensions'][0]
                
                delete_keys = ('name', 'category', 'creationTime', 'viewed', 'account', 'extensions')
                for key in delete_keys:
                    if key in processed_doc:
                        processed_doc.pop(key, None)  # безопасное удаление
                
                doc_dict[account].append(processed_doc)
                
    return doc_dict

In [ ]:
async def dowload_all_documents(account: str, api_token: str, session: aiohttp.ClientSession, docs_for_download: dict = None):
    """Функция для скачивания всех документов, переданных в docs_for_download.
    """
    if docs_for_download is None:
        docs_for_download = await processing_doc_list_wb()

    url = "https://documents-api.wildberries.ru/api/v1/documents/download/all"
    headers = {
        "Authorization": api_token
    }
    params = {'params': []}

    # Выделяем список документов для конкретного аккаунта
    params['params'].extend(docs_for_download.get(account, []))

    # Интервал для запросов на ВБ
    delay = 350
    # Количество попыток в случае ошибки
    attempts = 5
    for _ in range(attempts+1):
        try:
            async with session.post(url, headers=headers, json=params, timeout=15) as res:
                    # 1. Пытаемся распарсить JSON, если это возможно
                    content_type = res.headers.get('Content-Type', '')
                    data = await res.json() if 'application/json' in content_type else None

                    # Статусы при которых инициируем повторный запрос
                    delay_statuses = (421, 400, 503)
                    error_detail = data.get('detail') if data else await res.text()

                    if res.status == 200:
                        data['account'] = account    
                        print(f"✅ [{account}] Данные получены по кабинету {account}")
                        return data
                    elif res.status in delay_statuses:
                        print(f'❓ Ошибка {error_detail}')
                        asyncio.sleep(delay)
                    else:
                         print(f'❌ [{account}] Ошибка {res.status}: {error_detail}')
                    
                    return None
        
        except aiohttp.ClientError as e:
                print(f"💥 [{account}] Ошибка сессии: {e}")
                return None
        except asyncio.TimeoutError as e:
                print(f"💥 [{account}] Таймаут: {e}")
                asyncio.sleep(delay) 
        except Exception as e:
                print(f"💥 [{account}] Непредвиденная ошибка: {e}")
                return None                
        

async def fetch_download_all_documents(tokens: dict = load_api_tokens()):
      """ Асинхронная функция для скачивания запрашиваемых документов со всех доступных аккаунтов"""
      async with aiohttp.ClientSession() as session:
        # Создаем задачи для каждого аккаунта и токена
        tasks = [dowload_all_documents(account, api_token, session) for account, api_token in tokens.items()]
        # Ожидаем завершения всех задач и собираем результаты
        results = await asyncio.gather(*tasks)
        return results


In [14]:
docs = await fetch_download_all_documents()

⏳ [СТАРТ9237] Ошибка 429: {
    "title": "too many requests",
    "detail": "Limited by global limiter, per seller 1e3cf193-6d69-4a22-8c86-efad1bb4d516; See https://dev.wildberries.ru/openapi/api-information",
    "code": "461a0b83d6bd 2950e93b5fda",
    "requestId": "117e742a61c0c825a598c7e2742b9bfd",
    "origin": "s2s-documents",
    "status": 429,
    "statusText": "Too Many Requests",
    "timestamp": "2026-02-24T13:17:35Z"
}
. Повтор через 10 сек.
⏳ [Оганесян] Ошибка 429: {
    "title": "too many requests",
    "detail": "Limited by global limiter, per seller 1de3c6ac-5f82-4a3d-b0f8-462a557cf8d7; See https://dev.wildberries.ru/openapi/api-information",
    "code": "461a0b83d6bd 2950e93b5fda",
    "requestId": "332375423689956ecc77e9450fbc54e3",
    "origin": "s2s-documents",
    "status": 429,
    "statusText": "Too Many Requests",
    "timestamp": "2026-02-24T13:17:35Z"
}
. Повтор через 10 сек.
⏳ [Оганесян] Ошибка 429: {
    "title": "too many requests",
    "detail": "Limited b

CancelledError: 

In [7]:
import base64
import zipfile
import io

def inspect_archive(zip_bytes: bytes, indent: str = "", is_top: bool = True):
    """Рекурсивно обходит ZIP-архив и выводит содержимое.
    Если внутри есть файлы .zip, распаковывает и их."""
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        for name in zf.namelist():
            print(f"{indent}📄 {name}")
            with zf.open(name) as f:
                data = f.read()
                # Если внутри ещё один ZIP-архив
                if name.lower().endswith('.zip') and data.startswith(b'PK'):
                    print(f"{indent}  └─ (вложенный архив)")
                    inspect_archive(data, indent + "    ", is_top=False)
                # Если текстовый файл – покажем первые строки
                elif any(name.lower().endswith(ext) for ext in ['.csv', '.json', '.txt']):
                    try:
                        text = data.decode('utf-8')
                        lines = text.splitlines()
                        print(f"{indent}    Первые 5 строк:")
                        for line in lines[:5]:
                            print(f"{indent}      {line[:100]}...")
                    except UnicodeDecodeError:
                        print(f"{indent}    (бинарный файл, не показываем)")


# for item in docs_all:
#     if item:
#         account = item['account']
#         data = item['data']
#         file_name = data['fileName']
#         zip_b64 = data['document']

#         print(f"\n{'='*60}")
#         print(f"Аккаунт: {account}")
#         print(f"Файл: {file_name}")

#         try:
#             zip_bytes = base64.b64decode(zip_b64)
#             inspect_archive(zip_bytes)
#         except Exception as e:
#             print(f"Ошибка при обработке архива: {e}")

In [8]:
def extract_all_files(zip_bytes, base_path=""):
    """Рекурсивно извлекает все файлы из ZIP (и вложенных ZIP) в словарь вида {имя_файла: байты}."""
    result = {}
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        for name in zf.namelist():
            full_name = f"{base_path}/{name}" if base_path else name
            with zf.open(name) as f:
                data = f.read()
                # Если это ZIP, рекурсивно заходим внутрь
                if name.lower().endswith('.zip') and data.startswith(b'PK'):
                    inner = extract_all_files(data, full_name)
                    result.update(inner)
                else:
                    result[full_name] = data
    return result


async def processed_zip_docs(docs_all=None):
    """Проходим по всем скачанным архивам.
    Функция extract_files_from_zip обрабатывает все архивированные файлы внутри скачанного архива. Возвращаем список словарей, где где ключ 'bytes' содержит байты конечных файлов (уже не архивов)"""
    if docs_all is None:
        docs_all = await fetch_download_all_documents()
    
    all_files = []  # список всех конечных файлов со всех документов
    for doc in docs_all:
        if doc:
            account = doc['account']
            doc_bytes = base64.b64decode(doc['data']['document'])
            print(type(doc_bytes))
            files = extract_files_from_zip(doc_bytes)
            # Добавим информацию об аккаунте в каждый файл
            for f in files:
                f['account'] = account
            all_files.extend(files)
    return all_files

def extract_files_from_zip(zip_bytes, base_path=""):
    """
    Рекурсивно извлекает все конечные файлы из ZIP-архива.
    Возвращает список словарей: {'path': полный_путь, 'bytes': байты_файла}
    """
    result = []
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        for name in zf.namelist():
            full_path = f"{base_path}/{name}" if base_path else name
            # Читаем содержимое
            data = zf.read(name)
            # Если это ZIP, рекурсивно обрабатываем
            if name.lower().endswith('.zip') and data.startswith(b'PK'):
                # Рекурсивный вызов
                result.extend(extract_files_from_zip(data, full_path))
            else:
                # Конечный файл
                result.append({'path': full_path, 'bytes': data})
    return result

In [9]:
processed_zip_files = await processed_zip_docs()

pdf_files = [f for f in processed_zip_files if f['path'].endswith('.pdf')]
xlsx_files = [f for f in processed_zip_files if f['path'].endswith('.xlsx')]
xml_files = [f for f in processed_zip_files if f['path'].endswith('.xml')]

⏳ [Старт] Ошибка 429: {
    "title": "too many requests",
    "detail": "Limited by global limiter, per seller 9ee4946c-6ea7-4f11-adab-d4368f8eeada; See https://dev.wildberries.ru/openapi/api-information",
    "code": "461a0b83d6bd 2950e93b5fda",
    "requestId": "4f88e6f665857d22177b25ac797c554f",
    "origin": "s2s-documents",
    "status": 429,
    "statusText": "Too Many Requests",
    "timestamp": "2026-02-24T07:27:21Z"
}
. Повтор через 10 сек.
⏳ [Старт] Ошибка 429: {
    "title": "too many requests",
    "detail": "Limited by global limiter, per seller 9ee4946c-6ea7-4f11-adab-d4368f8eeada; See https://dev.wildberries.ru/openapi/api-information",
    "code": "461a0b83d6bd 2950e93b5fda",
    "requestId": "ae8318554f5a73aedb03109cf106dde1",
    "origin": "s2s-documents",
    "status": 429,
    "statusText": "Too Many Requests",
    "timestamp": "2026-02-24T07:27:21Z"
}
. Повтор через 10 сек.
⏳ [Старт] Ошибка 429: {
    "title": "too many requests",
    "detail": "Limited by global l

C:\Users\123\AppData\Local\Temp\ipykernel_1468\307672653.py:37: RuntimeWarning: coroutine 'sleep' was never awaited
  asyncio.sleep(delay)


✅ [Хачатрян] Данные получены по кабинету Хачатрян
❌ [Даниелян] Не удалось получить данные для категории redeem-notification
⏳ [Оганесян] Ошибка 429: {
    "title": "too many requests",
    "detail": "Limited by global limiter, per seller 1de3c6ac-5f82-4a3d-b0f8-462a557cf8d7; See https://dev.wildberries.ru/openapi/api-information",
    "code": "461a0b83d6bd 2950e93b5fda",
    "requestId": "6cb02e861881a22af7170ad56fce53f9",
    "origin": "s2s-documents",
    "status": 429,
    "statusText": "Too Many Requests",
    "timestamp": "2026-02-24T07:29:59Z"
}
. Повтор через 10 сек.
⏳ [Вектор2] Ошибка 429: {
    "title": "too many requests",
    "detail": "Limited by global limiter, per seller c9f14cb0-ece5-498d-8a0c-395846549dbb; See https://dev.wildberries.ru/openapi/api-information",
    "code": "461a0b83d6bd 2950e93b5fda",
    "requestId": "01c1e69bd8cf3ad3a9cabc06fbc65721",
    "origin": "s2s-documents",
    "status": 429,
    "statusText": "Too Many Requests",
    "timestamp": "2026-02-24

In [10]:
xlsx_files = [f for f in processed_zip_files if f['path'].endswith('.xlsx')]

In [11]:
import pdfplumber
import pandas as pd

def extract_table_from_pdf(pdf_bytes):
    """
    Извлекает таблицу из PDF (первая страница).
    Возвращает список списков или None, если таблица не найдена.
    """
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        # Предполагаем, что таблица на первой странице
        page = pdf.pages[0]
        # Пробуем extract_table() — он ищет одну таблицу
        table = page.extract_table()
        if table:
            return table
        # Если не сработало, попробуем extract_tables() и возьмём первую
        tables = page.extract_tables()
        if tables:
            return tables[0]
    return None


# Функция для очистки числового значения
def clean_number(val):
    if pd.isna(val) or val in ('—', 'X', ''):
        return 0.0
    # Удаляем пробелы, заменяем запятую на точку
    val = str(val).replace(' ', '').replace(',', '.')
    try:
        return float(val)
    except ValueError:
        return 0.0

def processed_pdf_week_reps(pdf_files: list):
    """Функция обрабатывает pdf файлы, переданные списком в битовой формате и возвращает единый датафрейм. На вход ожидается список словарей с ключами bytes и account"""
    # Список для хранения извлеченных данных в виде таблиц
    pdf_tables = []
    for item in pdf_files:
        # Для каждого файла извлекаем битовые данные
        pdf_bytes = item['bytes']
        # Отдельно выносим аккаунт
        account = item['account']
        # Из pdf файла извлекаем таблицу
        table = extract_table_from_pdf(pdf_bytes)
        # Если таблица существует приводим ее к датафрейму для дальнейше обработки
        if table:
            df = pd.DataFrame(table[1:], columns=table[0])
            # Приводим числовые столбцы к float, убираем пробелы и заменяем запятую на точку
            if 'Сумма, руб.' in df.columns:
                df['Сумма, руб.'] = df['Сумма, руб.'].apply(clean_number)
            if 'в т.ч НДС, руб.' in df.columns:
                df['в т.ч НДС, руб.'] = df['в т.ч НДС, руб.'].apply(clean_number)
            if 'Дата' in df.columns:
                df['Дата'] = pd.to_datetime(df['Дата']).dt.date
            # Добавляем данные об аккаунте
            df['account'] = account.upper()
            pdf_tables.append(df)

    if pdf_tables:
        return pd.concat(pdf_tables, ignore_index=True)
    else:
        return pd.DataFrame() 

In [12]:
from core.utils_sql import get_db_engine, sync_data_to_postgres
from sqlalchemy import BigInteger, String, Boolean, DateTime, Numeric, Float

# Достаем все еженедельные отчеты в виде датафрейма
week_reps_data = processed_pdf_week_reps(pdf_files)

engine = get_db_engine()

weekly_implementation_report_dict = {
    "weekly_implementation_report": {
        '№': String(255),
        'Наименование': String(255),
        'Документ основание': String(255),
        'Дата': DateTime(),
        '№ документа': Numeric(12,2),
        'Сумма, руб.': Numeric(12,2),
        'в т.ч НДС, руб.': Numeric(12,2),
        'account': String(255)
    }
}

table_name = list(weekly_implementation_report_dict.keys())[0]
unique_keys = ['№ документа', '№', 'account']
schema_definition = weekly_implementation_report_dict[table_name]


sync_data_to_postgres(engine, table_name, week_reps_data.to_dict('records'), schema_definition, unique_keys)

✅ Таблица 'weekly_implementation_report': успешно синхронизирована по ключам ['№ документа', '№', 'account']
✅ Таблица 'weekly_implementation_report': обработано 572 строк.


In [13]:
from openpyxl import load_workbook
redeems_list = []
for item in xlsx_files:
    # Для каждого файла извлекаем битовые данные
    xlsx_bytes = item['bytes']
    # Отдельно выносим аккаунт
    account = item['account']
    # Из xlsx файла извлекаем таблицу
    xlsx_file = load_workbook(io.BytesIO(xlsx_bytes), read_only=True)
    file_sheet = xlsx_file.active
    doc_name = file_sheet['A3'].value
    xlsx_data = list(file_sheet.iter_rows(values_only=True))
    df = pd.DataFrame(xlsx_data[10:], columns=xlsx_data[9])
    if doc_name:
        df['Документ'] = doc_name
    df['account'] = account.upper()
    # # header = None
    # for row in file_sheet.iter_rows(values_only=True):
    #     print(row)
    redeems_list.append(df)


# if pdf_tables
#     return pd.concat(pdf_tables, ignore_index=True)
# else:
#     return pd.DataFrame() 

In [14]:
def process_redeem_file(item: dict):
    """
    Вспоомгательная функция.
    Обрабатывает один Excel-файл с уведомлением о выкупе.
    Возвращает DataFrame с данными таблицы и метаданными.
    """
    xlsx_bytes = item['bytes']
    account = item['account']
    
    # Загружаем workbook
    wb = load_workbook(io.BytesIO(xlsx_bytes), read_only=True)
    sheet = wb.active
  
    # Загружаем все строки в список
    rows = list(sheet.iter_rows(values_only=True))
    
    # Находим начало таблицы (где заголовки)
    table_start = None
    for i, row in enumerate(rows):
        if row[0] and '№' in str(row[0]) and ('п/п' in str(row[0]) or 'Артикул' in str(row[1])):
            table_start = i
            break
    
    if table_start is None:
        # Если не нашли, используем стандартную 9-ю строку (0-based)
        table_start = 9
    
    # Заголовки таблицы
    headers = rows[table_start]
    # Данные таблицы (начиная со следующей строки)
    data_rows = rows[table_start+1:]
    
    # Создаём DataFrame
    df = pd.DataFrame(data_rows, columns=headers)
    
    # Убираем строку с итогами (где в первой колонке 'Итого:')
    df = df[df.iloc[:, 0] != 'Итого:']
    
    # Преобразуем числовые колонки
    # Индексы колонок: 0 - №, 1 - Артикул, 2 - Наименование, 3 - Количество,
    # 4 - Сумма выкупа, 5 - Ставка НДС, 6 - Сумма НДС, 7 - КИЗ
    
    df.iloc[:, 3] = pd.to_numeric(df.iloc[:, 3], errors='coerce').fillna(0)
    df.iloc[:, 4] = df.iloc[:, 4].apply(clean_number)
    df.iloc[:, 6] = df.iloc[:, 6].apply(clean_number)
    
    # Добавляем метаданные
    doc_name = sheet['A3'].value
    df['Документ'] = doc_name
    df['account'] = account

    # Удаляем то, что снизу после таблицы в файле
    # 1. Находим индекс итоговой строки
    total_idx = None
    for i, val in enumerate(df.iloc[:, 0]):
        if val == 'Итого:':
            total_idx = i
            break

    # 2. Обрезаем DataFrame до итогов (если итоги найдены)
    if total_idx is not None:
        df = df.iloc[:total_idx]
    df = df[pd.to_numeric(df.iloc[:, 0], errors='coerce').notna()]
    
    return df

In [15]:
def process_xlsx_redeem_file(xlsx_files: dict):
    """
    Обрабатывает список Excel-файлов с уведомлениями о выкупе и возвращает объединённый DataFrame.
    
    Параметры:
    -----------
    xlsx_files : dict
        Список словарей, каждый из которых содержит:
        - 'bytes': байтовое содержимое Excel-файла
        - 'path': путь к файлу внутри архива
        - 'account': название аккаунта
        
    Возвращает:
    -----------
    pd.DataFrame
        Объединённый DataFrame со всеми записями из всех файлов,
        с корректными типами данных и без пустых колонок.
    """
    
    # Список для накопления DataFrame'ов из каждого файла
    all_reedems = []
    
    # Перебираем все переданные файлы
    for item in xlsx_files:
        try:
            # Обрабатываем отдельный файл с помощью вспомогательной функции
            df = process_redeem_file(item)
            # Добавляем результат в общий список
            all_reedems.append(df)
            print(f"✅ Обработан {item['path']}, строк: {len(df)}")
        except Exception as e:
            # Логируем ошибку, но не прерываем обработку остальных файлов
            print(f"❌ Ошибка в {item['path']}: {e}")

    # Если хотя бы один файл успешно обработан
    if all_reedems:
        # Объединяем все DataFrame'ы в один
        # ignore_index=True обеспечивает сквозную нумерацию строк
        df_reedems = pd.concat(all_reedems, ignore_index=True)

        # Удаляем колонки с пустыми именами
        # (например, если в исходном файле были лишние пустые столбцы)
        df_reedems = df_reedems.loc[:, ~df_reedems.columns.isin([None, ''])]

        # Приведение числовых колонок к типу float
        # Список колонок, которые должны содержать числа
        float_cols = ['Количество', 'Сумма выкупа, руб., \n(вкл. НДС)', 'Сумма НДС*,\nРуб.']
        
        for col in df_reedems.columns:
            # Если колонка есть в списке числовых - преобразуем её в float
            if col in float_cols:
                df_reedems[col] = df_reedems[col].astype(float)


        df_reedems = df_reedems.rename(columns={"№ \nп/п": "№",
                                                "Артикул": "wild",
                                                "Наименование ": "subject_name",
                                                "Количество": "quantity",
                                                "Сумма выкупа, руб., \n(вкл. НДС)": "sum_rub_with_vat",
                                                "Ставка НДС*,\n%": "vat_rate",
                                                "Сумма НДС*,\nРуб.": "vat_sum_rub",
                                                "КИЗ": "kiz",
                                                "Документ": "doc_name",
                                                })
        # Регулярное выражение для извлечения нужной части
        pattern = r'(wild\d+)'
        # Используем метод str.extract для извлечения нужного паттерна
        df_reedems['wild'] = df_reedems['wild'].str.extract(pattern)
        # Возвращаем итоговый DataFrame
        return df_reedems
    
    # Если ни один файл не обработан, возвращаем пустой DataFrame
    return pd.DataFrame()       

In [16]:
from core.utils_sql import get_db_engine, sync_data_to_postgres
from sqlalchemy import BigInteger, String, Boolean, DateTime, Numeric, Float

# Достаем все еженедельные отчеты в виде датафрейма
redeem_file_data = process_xlsx_redeem_file(xlsx_files)

engine = get_db_engine()

redeem_notification_dict = {
    "redeem_notification": {
        '№': String(255),
        'wild': String(255),
        'subject_name': String(255),
        'quantity': BigInteger,
        'sum_rub_with_vat': Numeric(12,2),
        'vat_rate': String(10),
        'vat_sum_rub': Numeric(12,2),
        'kiz': String(255),
        'doc_name': String(255),
        'account': String(255)
    }
}

table_name = list(redeem_notification_dict.keys())[0]
unique_keys = ['doc_name', '№', 'account']
schema_definition = redeem_notification_dict[table_name]

sync_data_to_postgres(engine, table_name, redeem_file_data.to_dict('records'), schema_definition, unique_keys)

✅ Обработан redeem-notification-628078646.zip/Уведомление о выкупе №628078646 от 2026-02-09.xlsx, строк: 31
✅ Обработан redeem-notification-620986776.zip/Уведомление о выкупе №620986776 от 2026-02-02.xlsx, строк: 40
✅ Обработан redeem-notification-615070056.zip/Уведомление о выкупе №615070056 от 2026-01-26.xlsx, строк: 37
✅ Обработан redeem-notification-628182186.zip/Уведомление о выкупе №628182186 от 2026-02-09.xlsx, строк: 35
✅ Обработан redeem-notification-620934280.zip/Уведомление о выкупе №620934280 от 2026-02-02.xlsx, строк: 44
✅ Обработан redeem-notification-615054494.zip/Уведомление о выкупе №615054494 от 2026-01-26.xlsx, строк: 48
✅ Обработан redeem-notification-627751002.zip/Уведомление о выкупе №627751002 от 2026-02-09.xlsx, строк: 67
✅ Обработан redeem-notification-621654034.zip/Уведомление о выкупе №621654034 от 2026-02-02.xlsx, строк: 53
✅ Обработан redeem-notification-614356313.zip/Уведомление о выкупе №614356313 от 2026-01-26.xlsx, строк: 58
✅ Обработан redeem-notificat

In [17]:
redeem_file_data.columns

Index(['№', 'wild', 'subject_name', 'quantity', 'sum_rub_with_vat', 'vat_rate',
       'vat_sum_rub', 'kiz', 'doc_name', 'account'],
      dtype='object')